# Day 44 — Model deployment basics: save model & FastAPI
Objectives:
- Save/load models (joblib).
- Build a minimal FastAPI endpoint.
- Send a sample request.
Note: Running the server requires a terminal (see code comments).


In [ ]:
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

artifact_dir = Path('artifacts/day44')
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / 'model.joblib'
X,y = load_iris(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,random_state=42)
clf = LogisticRegression(max_iter=1000).fit(Xtr,ytr)
print('test acc:', clf.score(Xte,yte))
joblib.dump(clf, model_path)


## Minimal FastAPI app (app.py)
Create a file `app.py` in this folder:
```python
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()
model = joblib.load(model_path)

class IrisFeatures(BaseModel):
    features: list

@app.post('/predict')
def predict(data: IrisFeatures):
    X = np.array([data.features])
    pred = model.predict(X).tolist()[0]
    return {'prediction': int(pred)}
```
Run server:
```bash
uvicorn app:app --reload
```
Send a request (new terminal):
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"features": [5.1, 3.5, 1.4, 0.2]}'
```


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — validated request contracts, trusted artifacts, and testable inference endpoints

### Mental model

Deployment turns a model into a boundary other software can call. The
request schema validates untrusted JSON before it reaches model code;
the feature builder establishes exact name, order, type, missing-value,
and range rules; the response schema makes outputs stable for clients.

FastAPI handles HTTP routing and delegates data validation to Pydantic.
A successful model load is not proof of compatibility or trust.
Artifact format, source, digest, library versions, and feature schema
must be verified before the service becomes ready.

### Read the API before running it

- **`class Request(BaseModel)`:** declares typed input fields and validation constraints that become JSON Schema.
- **`@app.post('/predict', response_model=...)`:** binds an HTTP method/path to a validated function contract.
- **`TestClient(app).post(..., json=payload)`:** exercises serialization, routing, validation, and response behavior without starting a network server.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — make request validation fail before inference

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Exactly four ordered measurements is the public contract; coercion and range policy are documented.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class PredictionRequest(BaseModel):
    measurements: list[float] = Field(min_length=4, max_length=4)

valid = PredictionRequest.model_validate(
    {"measurements": [5.1, 3.5, 1.4, 0.2]}
)
print(valid.model_dump())
try:
    PredictionRequest.model_validate({"measurements": [5.1, 3.5]})
except ValidationError as exc:
    print(exc.errors()[0]["type"])

**Expected observation:** The valid payload becomes a typed object; the short list raises a structured validation error before any model call.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — check artifact feature compatibility explicitly

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Feature count alone is insufficient in production; names, order, types, and preprocessing version must also match.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = np.array([0, 0, 1, 1])
model = LogisticRegression().fit(X, y)

request_features = np.array([[0.2, 0.8]])
if request_features.shape[1] != model.n_features_in_:
    raise ValueError("feature-count mismatch")
print(model.predict_proba(request_features).tolist())

**Expected observation:** Inference proceeds only after the request matrix matches the fitted feature-count contract.

### Debugging and practice ramp

**Common mistake:** Loading an arbitrary pickle/joblib file or accepting a raw list with undocumented feature order.

**Diagnostic:** Test valid, missing, extra, wrong-type, non-finite, out-of-range, and batch-size payloads; log only bounded non-sensitive metadata.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define validated request contracts, trusted artifacts, and testable inference endpoints in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not bind a development server publicly or treat a local endpoint as production-ready security.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Add input validation and friendly error behavior.

**Verify:** For task `Add input validation and friendly error behavior`, demonstrate the concrete requirement “1. Add input validation and friendly error behavior” with explicit inputs, observable output, and one counterexample.






2. Return the class name as well as the numeric class identifier.

**Verify:** For task `Return the class name as well as the numeric class identifier`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior.






3. Create a minimal runtime dependency file for this API.

**Verify:** For task `Create a minimal runtime dependency file for this API`, record the exact command/input, terminal result or returned value, and repeat the critical check from a clean process or fresh state.







### Progressive hints

1. Let Pydantic reject the wrong length or nonnumeric values. Do not catch every
   exception and turn programming defects into vague client errors.
2. Keep the mapping beside the model metadata and test all valid numeric class
   identifiers.
3. Include only direct runtime imports. Pin or lock versions through the
   repository tooling rather than copying the entire development environment.

### Additional mastery practice

Make an API boundary explicit: validate shape and meaning, map model outputs to a versioned schema, and test behavior without relying on a manually running server.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Boundary-case testing:** Write API tests for a missing feature, an extra feature, a string, NaN/infinity, wrong feature count, and one valid request. State the expected status-code family for each.
   **Progressive hint:** Use FastAPI TestClient so validation can be tested in-process. Malformed client input is 4xx; unexpected service failure is 5xx.

**Verify:** For task `Boundary-case testing: Write API tests for a missing feature, an extra feature, a string, NaN...`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then measure peak active/queued work, account for every input, and prove permits/resources are released after success and injected failure.







5. **Batch contract:** Design a `/predict-batch` request and response with stable row IDs, a maximum batch size, ordered results, and per-request model metadata.
   **Progressive hint:** Validate the entire batch before scoring or define explicit partial failure semantics. Never rely only on list position to identify rows.

**Verify:** For task `Batch contract: Design a /predict-batch request and response with stable row IDs, a maximum b...`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.







6. **Artifact-compatibility check:** At startup, validate model version, expected feature schema, and class metadata before accepting traffic. Explain why loading a pickle from an untrusted source is unsafe.
   **Progressive hint:** Persist a small manifest beside the artifact and compare required fields. Python pickle/joblib loading can execute code.

**Verify:** For task `Artifact-compatibility check: At startup, validate model version, expected feature schema, an...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Boundary-case testing


# Practice 5 — Batch contract


# Practice 6 — Artifact-compatibility check
